# Day 4

## Tokenizing with code

In [14]:
import tiktoken

encoding = tiktoken.encoding_for_model("gpt-4.1-mini")

tokens = encoding.encode("Hi my name is Ed and I like banoffee pie")

print(tokens)
print(f"Total tokens: {len(tokens)}")

[12194, 922, 1308, 382, 6117, 326, 357, 1299, 9171, 26458, 5148]
Total tokens: 11


In [15]:
import tiktoken

# Use a generic encoding if your model is Llama 3 / 8B or 70B
encoding = tiktoken.get_encoding("cl100k_base")

# Or map directly for the model you are using
# encoding = tiktoken.encoding_for_model("gpt-4") 

tokens = encoding.encode("Hi my name is Ed and I like banoffee pie")
print(tokens)
print(f"Total tokens: {len(tokens)}")

[13347, 856, 836, 374, 3279, 323, 358, 1093, 9120, 21869, 4447]
Total tokens: 11


In [16]:
tokens

[13347, 856, 836, 374, 3279, 323, 358, 1093, 9120, 21869, 4447]

In [17]:
for token_id in tokens:
    token_text = encoding.decode([token_id])
    print(f"{token_id} = {token_text}")

13347 = Hi
856 =  my
836 =  name
374 =  is
3279 =  Ed
323 =  and
358 =  I
1093 =  like
9120 =  ban
21869 = offee
4447 =  pie


In [18]:
encoding.decode([326])

' l'

# And another topic!

### The Illusion of "memory"

Many of you will know this already. But for those that don't -- this might be an "AHA" moment!

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")

### You should be very comfortable with what the next cell is doing!

_I'm creating a new instance of the OpenAI Python Client library, a lightweight wrapper around making HTTP calls to an endpoint for calling the GPT LLM, or other LLM providers_

In [19]:
from openai import OpenAI

# openai = OpenAI()
ollama_url = "http://127.0.0.1:11434/v1"
ollama_key = "anything"
openai = OpenAI(base_url=ollama_url, api_key=ollama_key)


### A message to OpenAI is a list of dicts

In [20]:
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "Hi! I'm Ed!"}
    ]

In [21]:
# response = openai.chat.completions.create(model="gpt-4.1-mini", messages=messages)
response = openai.chat.completions.create(model="llama3.2:1b", messages=messages)
response.choices[0].message.content

"Hello Ed! It's nice to meet you. How can I help you today?"

### OK let's now ask a follow-up question

In [22]:
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "What's my name?"}
    ]

In [24]:
# response = openai.chat.completions.create(model="gpt-4.1-mini", messages=messages)
response = openai.chat.completions.create(model="llama3.2:1b", messages=messages)

response.choices[0].message.content

'I don\'t have any information about your identity, and I don\'t know what you\'re referring to. This conversation started with your question "Who\'s your name?" which indicates that you might not have shared that information yet. My main goal is to provide helpful assistance, and I\'d be happy to start fresh if you\'d like to share your name or ask a different question.'

### Wait, wha??

We just told you!

What's going on??

Here's the thing: every call to an LLM is completely STATELESS. It's a totally new call, every single time. As AI engineers, it's OUR JOB to devise techniques to give the impression that the LLM has a "memory".

In [25]:
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "Hi! I'm Ed!"},
    {"role": "assistant", "content": "Hi Ed! How can I assist you today?"},
    {"role": "user", "content": "What's my name?"}
    ]

In [26]:
# response = openai.chat.completions.create(model="gpt-4.1-mini", messages=messages)
response = openai.chat.completions.create(model="llama3.2:1b", messages=messages)
response.choices[0].message.content

"I make sure to call you out when I don't already know your name. Your name is actually Ed, but I'm happy to chat with you as Ed if that's what you'd prefer. What brings you here today ?"

## To recap

With apologies if this is obvious to you - but it's still good to reinforce:

1. Every call to an LLM is stateless
2. We pass in the entire conversation so far in the input prompt, every time
3. This gives the illusion that the LLM has memory - it apparently keeps the context of the conversation
4. But this is a trick; it's a by-product of providing the entire conversation, every time
5. An LLM just predicts the most likely next tokens in the sequence; if that sequence contains "My name is Ed" and later "What's my name?" then it will predict.. Ed!

The ChatGPT product uses exactly this trick - every time you send a message, it's the entire conversation that gets passed in.

"Does that mean we have to pay extra each time for all the conversation so far"

For sure it does. And that's what we WANT. We want the LLM to predict the next tokens in the sequence, looking back on the entire conversation. We want that compute to happen, so we need to pay the electricity bill for it!

